In [1]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os
from pathlib import Path

from scipy.ndimage import label, find_objects

import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import gc

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
working_dir = Path().absolute()

file_path = '/scratch/ng72/ms5578/hw_files'
write_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/output'

In [3]:
yr_file = f"{file_path}/HW_EHF_2018_2019.nc"
EHF_ds = xr.open_dataset(yr_file,
                         engine='netcdf4',
                         chunks="auto")

Using 26 connectivity, following Reddy, Perkins-Kirkpatrick, Sharples (2019)

In [4]:
def filter_2d_objs(data):
    structure = np.ones((3,3),dtype=int)
    lon, lat, time = data.shape
    filtered = np.zeros_like(data)

    for t in range(time):
        # Label connected components in 2D slice
        labeled, num_features = label(data[:, :, t], structure=structure)
        slices = find_objects(labeled)

        for i, slc in enumerate(slices):
            if slc is None:
                continue
            component_mask = (labeled[slc] == (i + 1))
            size = np.sum(component_mask)
            if size >= 10:
                # Keep the component
                filtered_slice = filtered[:, :, t]
                filtered_slice[slc][component_mask] = data[:, :, t][slc][component_mask]

    return filtered

In [5]:
EHF_flags = EHF_ds.EHF_flag.values

structure = np.ones((3, 3, 3), dtype=int)
# structure = generate_binary_structure(rank=3, connectivity=2)

event_arr = filter_2d_objs(EHF_flags)

# Filtering <100 voxel heatwaves
event_arr, tot_events = label(event_arr, structure=structure)
component_sizes = np.bincount(event_arr.ravel())[1:]
valid_components = np.isin(event_arr, np.where(component_sizes >= 100)[0] + 1)
event_arr = event_arr * valid_components

# Labelling again to fix colours/numbers
event_arr, tot_events = label(event_arr, structure=structure)

EHF_ds = EHF_ds.assign(events=(['lat','lon','time'], event_arr))
events = EHF_ds.events

# del event_arr, valid_components, component_sizes, structure, EHF_flags

In [6]:
# # Plot histogram of component sizes

# component_sizes = np.bincount(event_arr.ravel())[1:]

# plt.hist(component_sizes, bins=[0, 10, 20, 50, 100, 200, 500, 1000], edgecolor='black')
# plt.title("Histogram of Connected Component Sizes")
# plt.xlabel("Component Size (voxels)")
# plt.ylabel("Frequency")
# plt.show()

In [12]:
def gen_cmap(num_labels):
    # Original continuous cmap
    colors = ['blue', 'cyan', 'green', 'yellow', 'orange', 'red', 'purple']
    base_cmap = mcolors.LinearSegmentedColormap.from_list("custom_cmap", colors, N=1024)
    base_cmap.set_bad(color='white')

    # Sample evenly from the continuous cmap
    sampled_colors = base_cmap(np.linspace(0, 1, num_labels))
    discrete_cmap = mcolors.ListedColormap(sampled_colors)
    norm = mcolors.BoundaryNorm(np.arange(num_labels + 1) - 0.5, num_labels)

    return discrete_cmap, norm

cmap, norm = gen_cmap(tot_events)

In [13]:
nmap = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/raw/nmap.csv")

In [14]:
def plot_event(array, cmap, norm, time, nmap):
    lat_min, lat_max = -45, -10
    lon_min, lon_max = 110, 155

    # Subset at time
    labeled_at_time = array.isel(time=time)
    masked = np.ma.masked_where(labeled_at_time == 0, labeled_at_time)

    # Get lat/lon
    lons = array.lon.values
    lats = array.lat.values
    lon2d, lat2d = np.meshgrid(lons, lats)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)

    mesh = ax.pcolormesh(lon2d,
                          lat2d, 
                          masked, 
                          cmap=cmap, 
                          norm=norm, 
                          shading='auto', 
                          transform=ccrs.PlateCarree())
    
    ax.scatter(nmap['Lon'],
               nmap['Lat'],
               color='black',
               s=25,
               marker="x",
               alpha=0.4,
               transform=ccrs.PlateCarree())

    objs = np.unique(array)

    cbar = fig.colorbar(mesh, ax=ax, ticks=objs, label="Heatwave Event ID")
    cbar.ax.set_yticklabels([str(i) for i in objs])

    time_str = pd.to_datetime(array.time.values[time]).strftime('%Y-%m-%d')
    ax.set_title(f"Heatwave Events on {time_str}")

    del masked, mesh, ax, labeled_at_time, lon2d, lat2d
    
    return fig

In [15]:
def plot_gens(array, time, nmap):
    lat_min, lat_max = -45, -10
    lon_min, lon_max = 110, 155

    labeled_at_time = array.isel(time=time)
    masked = np.ma.masked_where(labeled_at_time == 0, labeled_at_time)

    lons = array.lon.values
    lats = array.lat.values
    lon2d, lat2d = np.meshgrid(lons, lats)

    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)

    ax.scatter(nmap['Lon'],
               nmap['Lat'],
               color='black',
               s=25,
               marker="x",
               alpha=0.4,
               transform=ccrs.PlateCarree())

    gen1 = ax.scatter(nmap[nmap['DUID'] == 'STWF1']['Lon'],
                      nmap[nmap['DUID'] == 'STWF1']['Lat'],
                      color='red',
                      s=400,
                      marker="*",
                      transform=ccrs.PlateCarree(),
                      label='Silverton Wind Farm (NSW)')

    gen2 = ax.scatter(nmap[nmap['DUID'] == 'MEWF1']['Lon'],
                      nmap[nmap['DUID'] == 'MEWF1']['Lat'],
                      color='blue',
                      s=400,
                      marker="*",
                      transform=ccrs.PlateCarree(),
                      label='Mount Emerald Wind Farm (QLD)')

    ax.legend(loc='lower left', title="Generators")

    # time_str = pd.to_datetime(array.time.values[time]).strftime('%Y-%m-%d')
    ax.set_title(f"Generators of the NEM")

    del masked, ax, labeled_at_time, lon2d, lat2d

    return fig

# fig = plot_gens(EHF_ds['EHF_flag'], 35, nmap)
# fig.show()
# fig.savefig(f"{write_path}/gen_locs_example.png")

In [ ]:
# Plot all hws on 3d scatter

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Get the coordinates of non-zero points (components)
dim3_arr = np.transpose(event_arr)
x, y, z = np.nonzero(dim3_arr)

# Scatter plot of the labeled components
ax.scatter(x, y, z, c=dim3_arr[x, y, z], cmap=cmap, marker='o', vmin=0, vmax=tot_events)

# Add colorbar
plt.colorbar(ax.scatter(x, y, z, c=dim3_arr[x, y, z], cmap=cmap, marker='o'))

plt.savefig(f"{dirpath}/all_hws.png")

In [ ]:
# slices = find_objects(events)
# labeled_slices = [
#     (label_id + 1, slc) 
#     for label_id, slc in enumerate(slices) 
#     if slc is not None
# ]

# for obj_id, sl in enumerate(labeled_slices, start=1):
#     if sl[1] is  None:
#         continue
#     else:
#         sub_array = events[:,:,sl[1][2]]

#         sdate = pd.to_datetime(sub_array.time.values[0]).strftime("%Y-%m-%d")
#         edate = pd.to_datetime(sub_array.time.values[-1]).strftime("%Y-%m-%d")
#         try:
#             dirname = f"HW_{sdate}_{edate}_ID_{obj_id}"
#             dirpath = f"{write_path}/{dirname}"
#             os.makedirs(dirpath)
#         except:
#             pass
        
#         for i in range(sub_array.shape[2]):
#             fig = plot_event(sub_array,cmap,norm,i,nmap)
#             fig.savefig(f"{dirpath}/day_{i}.png")
#             plt.close(fig)
#             plt.clf
#             gc.collect()

In [ ]:
def plot_year(array, nmap):
    lat_min, lat_max = -45, -10
    lon_min, lon_max = 110, 155

    masked = np.ma.masked_where(array == 0, array)

    # Get lat/lon
    lons = array.lon.values
    lats = array.lat.values
    lon2d, lat2d = np.meshgrid(lons, lats)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)

    mesh = ax.pcolormesh(lon2d,
                          lat2d, 
                          masked, 
                          cmap="Set2", 
                          shading='auto', 
                          transform=ccrs.PlateCarree())
    
    ax.scatter(nmap['Lon'],
               nmap['Lat'],
               color='black',
               s=25,
               marker="x",
               alpha=0.4,
               transform=ccrs.PlateCarree())

    time_str = pd.to_datetime(array.time.values).strftime('%Y-%m-%d')
    ax.set_title(f"Heatwave Events on {time_str}")

    del masked, mesh, ax, lon2d, lat2d
    
    return fig

EHF_flags = EHF_ds.EHF_flag

dirpath = f"{write_path}/EHF_2018_19"
os.makedirs(dirpath, exist_ok=True)

max_t = np.nonzero(EHF_flags.values)[2].max()
min_t = np.nonzero(EHF_flags.values)[2].min()

for t in range(min_t,max_t):
    fig = plot_year(EHF_flags[:,:,t],nmap)
    fig.savefig(f"{dirpath}/day_{t}.png")
    plt.close(fig)
    plt.clf
    gc.collect()

In [ ]:
plot_year(events,nmap)